# 00 — pandas for ML

The pandas you need to prepare data for a model: loading with the right types, selecting, missing values, group-by, joins,
reshaping, time series without leakage, fast code instead of `apply`, memory, and building a feature table. Everything runs on a small e-commerce dataset made in code.

## What's in this notebook

- 1\. Series, dataframe and the index
- 2\. The data: customers, products, orders
- 3\. Loading data with the right types
- 4\. A first look
- 5\. Selecting rows and columns
- 6\. Missing values
- 7\. Group-by: agg vs transform
- 8\. Merge and join
- 9\. Reshaping: pivot and melt
- 10\. Time series without leakage
- 11\. Vectorized code instead of `apply`
- 12\. Memory and speed
- 13\. Building a feature table for a model
- 14\. Trade-offs
- 15\. Interview questions
- 16\. In my own words

In [1]:
import warnings
warnings.filterwarnings("ignore", message=".*encountered in matmul", category=RuntimeWarning)

import io
import time
import numpy as np
import pandas as pd

print("pandas", pd.__version__)

pandas 2.3.3


# 1. SERIES, DATAFRAME AND THE INDEX

- A **Series** is one column: values plus an index (the row labels).
- A **DataFrame** is a table of Series that share one index. Each column has its own dtype.
- The **index** is used to align data: adding two Series matches them by label, not by position, and labels missing on one side give `NaN`.

In [2]:
price = pd.Series([10.0, 20.0, 30.0], index=["a", "b", "c"])
discount = pd.Series([1.0, 2.0], index=["b", "c"])
print(price - discount)

a     NaN
b    19.0
c    28.0
dtype: float64


"a" has no discount, so `price − discount` is `NaN` for it: pandas aligned the two Series by label. Use `price.sub(discount, fill_value=0)` when a missing value should count as 0.

# 2. THE DATA: CUSTOMERS, PRODUCTS, ORDERS

Three tables, as they would come from a shop's database: customers, products, and order lines.
Some prices are missing, and one product ID appears twice in the products table (a common data problem, used in section 8).

In [3]:
rng = np.random.default_rng(0)
n_customers, n_products, n_orders = 500, 40, 5000

customers = pd.DataFrame({
    "customer_id": [f"C{i:04d}" for i in range(n_customers)],
    "city": rng.choice(["Pune", "Mumbai", "Delhi", "Chennai", "Bengaluru"], n_customers),
    "signup_date": pd.Timestamp("2024-01-01") + pd.to_timedelta(rng.integers(0, 365, n_customers), unit="D"),
})
products = pd.DataFrame({
    "product_id": [f"P{i:03d}" for i in range(n_products)],
    "category": rng.choice(["electronics", "books", "fashion", "home"], n_products),
    "list_price": rng.uniform(100, 5000, n_products).round(0),
})
products = pd.concat([products, products.iloc[[5]].assign(list_price=999.0)], ignore_index=True)   # P005 appears twice

orders = pd.DataFrame({
    "order_id": np.arange(n_orders),
    "customer_id": rng.choice(customers["customer_id"].iloc[:450], n_orders),       # the last 50 customers never order
    "product_id": rng.choice(products["product_id"].unique(), n_orders),
    "order_date": pd.Timestamp("2025-01-01") + pd.to_timedelta(rng.integers(0, 365, n_orders), unit="D"),
    "quantity": rng.integers(1, 5, n_orders),
    "price": rng.uniform(100, 5000, n_orders).round(2),
})
orders.loc[rng.choice(n_orders, 150, replace=False), "price"] = np.nan

print(customers.shape, products.shape, orders.shape)
orders.head(3)

(500, 3) (41, 3) (5000, 6)


,order_id,customer_id,product_id,order_date,quantity,price
0,0,C0348,P030,2025-02-21,4,2320.72
1,1,C0282,P030,2025-02-22,3,3636.39
2,2,C0282,P014,2025-02-26,2,1951.15


# 3. LOADING DATA WITH THE RIGHT TYPES

`read_csv` guesses each column's type from the text. The guesses cost memory and correctness:
dates stay as plain text, IDs like "00123" lose their leading zeros, and repeated strings are stored as separate Python objects.
Tell pandas the types: `dtype=`, `parse_dates=`, and `usecols=` to load only the columns you need.

In [4]:
csv_text = orders.assign(zip_code="00123").to_csv(index=False)          # pretend this came from a file

guessed = pd.read_csv(io.StringIO(csv_text))
typed = pd.read_csv(io.StringIO(csv_text), parse_dates=["order_date"],
                    dtype={"customer_id": "category", "product_id": "category", "zip_code": "string"})

print("guessed:", dict(guessed.dtypes.astype(str)[["order_date", "zip_code", "customer_id"]]), " zip_code value:", guessed["zip_code"].iloc[0])
print("typed:  ", dict(typed.dtypes.astype(str)[["order_date", "zip_code", "customer_id"]]), " zip_code value:", typed["zip_code"].iloc[0])
print(f"memory: guessed {guessed.memory_usage(deep=True).sum() / 1e6:.2f} MB, typed {typed.memory_usage(deep=True).sum() / 1e6:.2f} MB")

guessed: {'order_date': 'object', 'zip_code': 'int64', 'customer_id': 'object'}  zip_code value: 123
typed:   {'order_date': 'datetime64[ns]', 'zip_code': 'string', 'customer_id': 'category'}  zip_code value: 00123
memory: guessed 1.11 MB, typed 0.53 MB


Without types, the dates load as text (`object`), the zip code "00123" becomes the number 123, and the table takes 1.11 MB.
With types, dates are real datetimes, the zip code keeps its zeros, and the IDs as `category` bring it down to 0.53 MB.

# 4. A FIRST LOOK

The first five calls on any new table: `shape`, `info()` (types, non-null counts), `describe()`, `isna().sum()`, and `value_counts()` on key columns.

In [5]:
orders.info()
print("\nmissing values per column:\n", orders.isna().sum()[lambda s: s > 0])
print("\norders per quantity:\n", orders["quantity"].value_counts().sort_index())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 6 columns):
 #   Column       Non-Null Count  Dtype         
---  ------       --------------  -----         
 0   order_id     5000 non-null   int64         
 1   customer_id  5000 non-null   object        
 2   product_id   5000 non-null   object        
 3   order_date   5000 non-null   datetime64[ns]
 4   quantity     5000 non-null   int64         
 5   price        4850 non-null   float64       
dtypes: datetime64[ns](1), float64(1), int64(2), object(2)
memory usage: 234.5+ KB

missing values per column:
 price    150
dtype: int64

orders per quantity:
 quantity
1    1242
2    1305
3    1216
4    1237
Name: count, dtype: int64


# 5. SELECTING ROWS AND COLUMNS

| Tool | Selects by | Example |
|---|---|---|
| `df["col"]`, `df[["a", "b"]]` | column names | `orders[["order_id", "price"]]` |
| `.loc[rows, cols]` | labels and boolean masks | `orders.loc[orders["quantity"] > 3, "price"]` |
| `.iloc[rows, cols]` | integer positions | `orders.iloc[:5, :2]` |
| `.query("...")` | an expression string | `orders.query("quantity > 3 and price < 500")` |

**Changing values:** always assign through `.loc` on the original table: `df.loc[mask, "col"] = value`.
Chained indexing like `df[mask]["col"] = value` may change a temporary copy and silently do nothing; pandas 2.x warns with `SettingWithCopyWarning`.
pandas 3.0 turns on **copy-on-write**, where any selection behaves as a copy, so chained assignment never changes the original.

In [6]:
big_orders = orders.loc[orders["quantity"] > 3, ["order_id", "customer_id", "price"]]
print(f"orders with quantity > 3: {len(big_orders)}")
print(f"same with query(): {len(orders.query('quantity > 3'))}")

orders_fixed = orders.copy()
orders_fixed["bulk"] = False
orders_fixed.loc[orders_fixed["quantity"] > 3, "bulk"] = True       # the right way to set values
print("rows marked bulk:", int(orders_fixed["bulk"].sum()))

orders with quantity > 3: 1237
same with query(): 1237
rows marked bulk: 1237


# 6. MISSING VALUES

- Find them: `isna().sum()`; look at **why** they are missing before choosing a fix.
- Drop: `dropna(subset=[...])` when few rows are affected and the rest are fine.
- Fill: `fillna(value)`, or fill **per group** with `groupby(...).transform("median")`, which is usually better than one global value.
- A missing-indicator column keeps the information that the value was missing (`13-ml-fundamentals/3-ml-feature-engineering.ipynb`).
- In a model pipeline, fit the imputer on the training rows only (a Pipeline does this), or the test rows leak into the fill values.

In [7]:
missing = orders["price"].isna()
filled_global = orders["price"].fillna(orders["price"].median())
filled_by_product = orders["price"].fillna(orders.groupby("product_id")["price"].transform("median"))
print(f"missing prices: {missing.sum()}")
print(f"one global median for all of them: {filled_global[missing].nunique()} distinct fill value(s)")
print(f"median of each product:            {filled_by_product[missing].nunique()} distinct fill values")

missing prices: 150
one global median for all of them: 1 distinct fill value(s)
median of each product:            38 distinct fill values


A global median fills all 150 gaps with the same number; filling with each product's own median gives 38 different values that follow the product.
(Here prices are random per order, so the gain is small; on real data where price depends on the product, it matters a lot.)

# 7. GROUP-BY: AGG VS TRANSFORM

- `groupby(...).agg(...)` returns **one row per group**: a summary table.
- `groupby(...).transform(...)` returns **one value per original row**: the group statistic broadcast back, ready to use as a feature.
- Named aggregation keeps the output columns readable: `agg(revenue=("amount", "sum"), orders=("order_id", "count"))`.

In [8]:
orders["amount"] = orders["quantity"] * orders["price"]

per_customer = orders.groupby("customer_id").agg(
    orders=("order_id", "count"),
    revenue=("amount", "sum"),
    avg_basket=("amount", "mean"),
).round(1)
print(per_customer.sort_values("revenue", ascending=False).head(3), "\n")

orders["customer_avg_amount"] = orders.groupby("customer_id")["amount"].transform("mean")
print("agg gives", per_customer.shape[0], "rows (one per customer); transform gives", orders["customer_avg_amount"].shape[0], "rows (one per order)")

             orders   revenue  avg_basket
customer_id                              
C0423            21  169772.9      8084.4
C0036            18  167699.9      9864.7
C0376            17  158779.0      9339.9 

agg gives 450 rows (one per customer); transform gives 5000 rows (one per order)


# 8. MERGE AND JOIN

| `how` | Keeps |
|---|---|
| `inner` | only keys found in both tables |
| `left` | every row of the left table; NaN where the right side has no match |
| `outer` | every key from both sides |

**The classic bug: duplicate keys multiply rows.** P005 appears twice in `products`, so every order of P005 matches two rows,
and the "joined" table quietly gains rows and double-counts revenue. Two guards:
- `validate="many_to_one"` raises an error if the right table's key isn't unique
- compare the row count before and after every join

In [9]:
joined = orders.merge(products, on="product_id", how="left")
print(f"orders: {len(orders)} rows -> after the join: {len(joined)} rows (P005 orders: {(orders['product_id'] == 'P005').sum()})")

try:
    orders.merge(products, on="product_id", how="left", validate="many_to_one")
except pd.errors.MergeError as e:
    print("validate caught it:", str(e)[:60])

products_clean = products.drop_duplicates(subset="product_id", keep="first")
joined = orders.merge(products_clean, on="product_id", how="left", validate="many_to_one")
print(f"after removing the duplicate: {len(joined)} rows")

check = customers.merge(orders[["customer_id"]].drop_duplicates(), on="customer_id", how="left", indicator=True)
print(f"customers with no orders: {(check['_merge'] == 'left_only').sum()}")

orders: 5000 rows -> after the join: 5127 rows (P005 orders: 127)
validate caught it: Merge keys are not unique in right dataset; not a many-to-on
after removing the duplicate: 5000 rows
customers with no orders: 50


The join turned 5,000 orders into 5,127 rows, one extra row for each of the 127 orders of P005. `validate="many_to_one"` stopped the bad join,
and after dropping the duplicate the row count is back to 5,000. `indicator=True` adds a `_merge` column, a quick way to find rows with no match.

# 9. RESHAPING: PIVOT AND MELT

- **Wide** tables have one column per category (good for reports and some models); **long** tables have one row per (entity, category, value) (good for group-by and plotting).
- `pivot_table` goes long → wide and aggregates duplicates; `melt` goes wide → long.
- `pd.crosstab` counts combinations of two columns.

In [10]:
with_category = orders.merge(products_clean[["product_id", "category"]], on="product_id", how="left")
wide = with_category.pivot_table(index="customer_id", columns="category", values="amount", aggfunc="sum", fill_value=0).round(0)
print("wide: one row per customer, one column per category", wide.shape)
print(wide.head(2), "\n")

long = wide.reset_index().melt(id_vars="customer_id", var_name="category", value_name="amount")
print("long again:", long.shape)

wide: one row per customer, one column per category (450, 4)
category       books  electronics  fashion     home
customer_id                                        
C0000        12888.0      22254.0  21803.0  21437.0
C0001        17166.0      15360.0   3801.0  34242.0 

long again: (1800, 3)


# 10. TIME SERIES WITHOUT LEAKAGE

- Make the date a real datetime (`pd.to_datetime`), then use `.dt.month`, `.dt.dayofweek`, ...
- `resample("MS")` groups by calendar period (needs a datetime index); `rolling(7)` is a moving window; `shift(1)` moves values one step later.
- **Leakage:** a rolling mean that includes today's value tells the model today's answer. For a feature that predicts day t, use only days before t:
  `shift(1)` first, then `rolling`.

In [11]:
daily = orders.set_index("order_date")["amount"].resample("D").sum().to_frame("revenue")
monthly = daily["revenue"].resample("MS").sum()
print("monthly revenue (first 3 months):\n", (monthly.head(3) / 1e6).round(2).rename("revenue, millions"), "\n")

daily["leaky_avg_7d"] = daily["revenue"].rolling(7).mean()               # includes today: leakage
daily["safe_avg_7d"] = daily["revenue"].shift(1).rolling(7).mean()       # only the 7 days before today
daily["lag_1"] = daily["revenue"].shift(1)
ok = daily.dropna()
print(f"correlation with today's revenue: leaky feature {ok['revenue'].corr(ok['leaky_avg_7d']):.2f}, safe feature {ok['revenue'].corr(ok['safe_avg_7d']):.2f}")

monthly revenue (first 3 months):
 order_date
2025-01-01    2.79
2025-02-01    2.38
2025-03-01    2.87
Freq: MS, Name: revenue, millions, dtype: float64 

correlation with today's revenue: leaky feature 0.35, safe feature -0.04


The leaky 7-day average has correlation 0.35 with the day's revenue because it contains it; the safe version, built only from earlier days, has −0.04.
A model trained with the leaky feature would look good offline and fail in production, where today's revenue isn't known yet.

# 11. VECTORIZED CODE INSTEAD OF `apply`

`df.apply(f, axis=1)` calls a Python function once per row, so it is a Python loop. Column operations, `np.where`, `np.select`,
`.str` and `.dt` methods run on whole columns at once.

In [12]:
def size_label(row):
    if row["amount"] > 10_000:
        return "large"
    if row["amount"] > 2_000:
        return "medium"
    return "small"


start = time.perf_counter()
slow = orders.apply(size_label, axis=1)
apply_s = time.perf_counter() - start

start = time.perf_counter()
fast = np.select([orders["amount"] > 10_000, orders["amount"] > 2_000], ["large", "medium"], default="small")
vec_s = time.perf_counter() - start

print(f"apply: {apply_s * 1000:.1f} ms   np.select: {vec_s * 1000:.2f} ms   -> about {apply_s / vec_s:,.0f}x faster, same result: {(slow.values == fast).all()}")

apply: 21.2 ms   np.select: 0.34 ms   -> about 62x faster, same result: True


On 5,000 rows `apply` takes 21.2 ms and `np.select` 0.34 ms, about 62x faster in this run. On millions of rows that is the difference between seconds and many minutes.

# 12. MEMORY AND SPEED

- `category` dtype stores each distinct string once plus small integer codes: big savings for columns with few distinct values.
- Downcast numbers where the range allows (`int64` → `int32`, `float64` → `float32`).
- Read big files in pieces: `read_csv(..., chunksize=100_000)`, or only the needed columns with `usecols`.
- When a table no longer fits comfortably in memory, or pandas is too slow, move to **Polars** (multi-threaded, lazy) or **DuckDB** (SQL on files), or Spark for clusters.

In [13]:
as_text = orders["customer_id"].astype(object)
as_category = orders["customer_id"].astype("category")
print(f"customer_id as text: {as_text.memory_usage(deep=True) / 1e3:.0f} KB   as category: {as_category.memory_usage(deep=True) / 1e3:.0f} KB")

customer_id as text: 310 KB   as category: 55 KB


# 13. BUILDING A FEATURE TABLE FOR A MODEL

The usual job: one row per customer, features from their history, and a label from **after** a cut-off date.
Features may only use data from before the cut-off; otherwise the model sees the future. This is the core of the P1 project.

In [14]:
cutoff = pd.Timestamp("2025-10-01")
history = orders[orders["order_date"] < cutoff]
future = orders[orders["order_date"] >= cutoff]

features = history.groupby("customer_id").agg(
    frequency=("order_id", "count"),
    monetary=("amount", "sum"),
    last_order=("order_date", "max"),
)
features["recency_days"] = (cutoff - features.pop("last_order")).dt.days
features["bought_again"] = features.index.isin(future["customer_id"]).astype(int)      # the label comes from after the cut-off

table = customers.set_index("customer_id")[["city"]].join(features, how="inner")
print(table.shape)
print(table.head(3))
print(f"\nshare of customers who bought again after the cut-off: {table['bought_again'].mean():.1%}")

(450, 5)
                  city  frequency  monetary  recency_days  bought_again
customer_id                                                            
C0000        Bengaluru          8  51071.77            16             1
C0001          Chennai         12  56861.71            13             1
C0002            Delhi          9  48782.28            23             1

share of customers who bought again after the cut-off: 93.1%


450 customers have history before the cut-off. Each row holds recency, frequency and monetary value (the classic **RFM** features) plus the city,
and the label says whether they ordered again after 1 October. Building the label from a different time window than the features is what keeps this table leakage-free.

# 14. TRADE-OFFS

| Decision | Options | What you gain and what you pay | Usual choice |
|---|---|---|---|
| Tool for tabular data | pandas / Polars / DuckDB (SQL) / Spark | pandas: everywhere, huge ecosystem, single-threaded and memory-hungry. Polars: much faster and multi-threaded, a different API. DuckDB: SQL on local files, very fast. Spark: clusters, heavy to run | pandas up to a few GB; Polars or DuckDB beyond that; Spark for cluster-scale data |
| Row logic | `apply` / vectorized | `apply` is easy to write but a Python loop; vectorized code is 10–1000x faster | vectorize; `apply` only for logic that can't be vectorized |
| Joining | `merge` / `map` | `merge` handles many columns and join types; `map` with a Series is quicker for looking up one column | `merge` with `validate=`; `map` for a single lookup |
| Table shape | wide / long | wide fits models and reports; long fits group-by, plotting and storage | keep long, pivot to wide at the end |
| Missing values | drop / fill / indicator | dropping loses rows; filling hides that a value was missing; an indicator keeps the signal | fill inside the Pipeline plus an indicator when missingness may mean something |

# 15. INTERVIEW QUESTIONS

### Q1. `loc` vs `iloc`?
`loc` selects by labels (index labels, column names, boolean masks); `iloc` selects by integer position. `df.loc[5]` is the row labelled 5, `df.iloc[5]` the sixth row.

### Q2. `agg` vs `transform` in a group-by?
`agg` returns one row per group (a summary); `transform` returns a value for every original row, the group statistic aligned back,
which is how you add a feature like "this customer's average order".

### Q3. What can go wrong with a merge?
Duplicate keys on the side you expect to be unique multiply rows and double-count values. Mismatched key types (int vs string) give no matches.
Guard with `validate="many_to_one"`, check row counts, and use `indicator=True` to see unmatched rows.

### Q4. What is `SettingWithCopyWarning`, and how do you avoid it?
pandas can't tell whether a chained selection like `df[mask]["col"] = x` changes `df` or a temporary copy. Assign with `df.loc[mask, "col"] = x`,
or call `.copy()` when you really want a separate table. pandas 3.0's copy-on-write makes every selection behave like a copy.

### Q5. How do you handle missing values?
First find out why they're missing. Then drop (few rows), fill (median, per-group median, or a model), and add a missing indicator when the fact of missing may matter.
In ML, fit the filling on training data only, inside a Pipeline.

### Q6. Why avoid `apply(axis=1)`?
It calls a Python function once per row, so it runs at Python-loop speed. Use column arithmetic, `np.where`, `np.select`, `.str` and `.dt`.

### Q7. How do you build a rolling feature without leakage?
Shift first, then roll: `s.shift(1).rolling(7).mean()` uses only the 7 previous periods. A plain `rolling(7)` includes the current value, which the model won't know at prediction time.

### Q8. How do you reduce a DataFrame's memory?
Load with explicit dtypes, convert low-cardinality strings to `category`, downcast numbers, load only needed columns, read in chunks, or switch to Polars or DuckDB.

### Q9. Coding task: top 3 products by revenue in each category.
`df.groupby(["category", "product_id"])["amount"].sum().reset_index().sort_values("amount", ascending=False).groupby("category").head(3)`

### Q10. Coding task: customers who never ordered.
A left merge with `indicator=True` and keep `_merge == "left_only"`, or `customers[~customers["customer_id"].isin(orders["customer_id"])]`.

### Q11. Coding task: 7-day rolling average of revenue per store.
Sort by date, then `df.groupby("store")["revenue"].transform(lambda s: s.rolling(7, min_periods=1).mean())`
(add `.shift(1)` first if it will be a model feature).

### Q12. pandas vs Polars vs SQL: when would you use each?
pandas for analysis and feature engineering on data that fits in memory; Polars when pandas is too slow or the data is a few GB;
SQL (in the warehouse, or DuckDB locally) when the data already lives in a database or to push joins and group-bys down to it.

# 16. IN MY OWN WORDS

Write this from memory, without looking above:

- `agg` vs `transform`:
- How a merge can silently duplicate rows, and how to catch it:
- How to build a rolling feature without leakage:
- How I'd make a slow pandas script faster: